# 02 · Quantization: from 32-bit floats to 8-bit integers

In this notebook you will:

1. see why 8-bit integers are attractive on a microcontroller;
2. learn the quantization formula $r \approx s\,(q - z)$ and use it by hand;
3. quantize our trained network to 8-bit integers, the way the Arduino will run it;
4. go below 8 bits, and keep the accuracy with **quantization-aware training**.

The notebook ends with a short overview of the other kinds of quantization you will meet in toolkits.

In [ ]:
import sys, pathlib, copy
# Only needed so Python can find the course folder `aiedge` (the helpers imported below).
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# EXTERNAL (shared with the other notebooks, code shown in notebook 01):
#   load_gesture_data  the synthetic sensor data (see the comment block in notebook 01)
#   MLP, count_params  the small network class and a parameter counter
#   train, accuracy, set_seed   the training loop, test accuracy, repeatable random numbers
# Why external: these are used in every notebook. Notebook 01 explains them line by
# line; repeating them here would only copy code. Everything about quantization,
# which is the topic of THIS notebook, is written out below.
# ---------------------------------------------------------------------------
from aiedge.data import load_gesture_data
from aiedge.models import MLP, count_params
from aiedge.training import train, accuracy, set_seed

First we re-create the data and the baseline from notebook 01. Same seed, so it is the same model; this takes a few seconds.

In [ ]:
data = load_gesture_data()
set_seed(0)
baseline = MLP(hidden=(64, 32))
train(baseline, data, augment=True, verbose=False)
base_acc = accuracy(baseline, data.X_test, data.y_test)
print(f"baseline (float32): test accuracy {base_acc:.1%}, {count_params(baseline):,} parameters "
      f"= {4 * count_params(baseline) / 1024:.1f} KB")

## 1. Why 8-bit integers?

A trained network is mostly a long list of numbers: the weights. Normally each one is a **float32**: a number with a decimal point, like 0.0372 or -1.25, stored in 32 bits (4 bytes).

An **int8** is a whole number stored in 8 bits (1 byte). It can only be one of the 256 values from **-128 to 127**.

Using int8 instead of float32 on a microcontroller gives:

1. **4x less memory**: 1 byte per weight instead of 4. For us this is the most important reason.
2. **Less energy**: an 8-bit integer multiplication costs about 20x less energy than a float32 multiplication.
3. **More speed on many chips**: chips without floating-point hardware (an *FPU*) do float maths in software, 10-100x slower. Our Cortex-M4F does have an FPU; in notebook 05 you measure what int8 brings there.

The question of this notebook: **how do we write numbers like 0.0372 or -1.25 as whole numbers from -128 to 127, without breaking the network?**

## 2. The quantization formula

We pick a **scale** $s$ (a small positive float) and a **zero-point** $z$ (an integer). A real number $r$ is represented by the integer

$$q = \text{clamp}\big(\text{round}(r / s) + z,\ q_{min},\ q_{max}\big)$$

and we can get back (approximately) the real number with

$$r \approx s \cdot (q - z)$$

Think of it as a ruler with 256 marks spaced $s$ apart: every real number is rounded to the nearest mark. The *rounding error* is at most $s/2$, half the distance between marks. The *clamp* cuts off numbers outside the ruler (*clipping error*).

### Symmetric quantization (for weights)
Weights are spread around 0 (both signs). We use $z = 0$ and pick $s$ so that the largest absolute weight lands on 127:

$$s = \frac{\max |w|}{127}$$

Let's do it by hand for four weights:

In [ ]:
w = torch.tensor([-0.62, 0.047, 0.313, 1.27])
s = w.abs().max() / 127                      # scale
q = torch.round(w / s).clamp(-127, 127)      # quantize   -> integers
w_back = s * q                               # dequantize -> approximately w

print(f"scale s = {s:.4f}")
print("original   :", w.numpy())
print("integers q :", q.int().numpy())
print("dequantized:", w_back.numpy().round(4))
print("error      :", (w_back - w).numpy().round(4), f"  (max possible: s/2 = {s / 2:.4f})")

### Asymmetric quantization (for activations)
After a ReLU all values are ≥ 0. Symmetric quantization would waste the whole negative half of the ruler (-128…-1). Instead we map the actual range $[lo, hi]$ onto $[-128, 127]$:

$$s = \frac{hi - lo}{255}, \qquad z = -128 - \text{round}(lo / s)$$

For a ReLU output with range $[0, 6]$: $s = 6/255 \approx 0.0235$ and $z = -128$. So real 0 ↦ -128, real 6 ↦ 127. Note that real **0 is represented exactly** (by $q = z$): this matters, because ReLU produces lots of exact zeros.

In [ ]:
def asymmetric_params(lo, hi, bits=8):
    """Scale and zero-point that map the real range [lo, hi] onto the integers -128...127."""
    lo, hi = min(float(lo), 0.0), max(float(hi), 0.0)   # the range must contain 0 (see above)
    qmin, qmax = -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
    scale = max(hi - lo, 1e-8) / (qmax - qmin)           # real distance between two marks
    zero_point = int(round(qmin - lo / scale))           # the integer that stands for real 0
    return scale, int(np.clip(zero_point, qmin, qmax))

scale, zp = asymmetric_params(0.0, 6.0)
a = torch.tensor([0.0, 0.01, 1.0, 3.3, 6.0])
q = torch.clamp(torch.round(a / scale) + zp, -128, 127)
print(f"scale = {scale:.4f}, zero-point = {zp}")
print("real       :", a.numpy())
print("integers q :", q.int().numpy())
print("dequantized:", (scale * (q - zp)).numpy().round(4))

### The range trade-off
The range must be chosen well:
* **too wide** (e.g. one huge outlier weight) → $s$ is large → all the *normal* values get coarse rounding;
* **too narrow** → large values are clipped.

This is why we often use one scale **per output neuron** ("per-channel") for weights instead of one scale for a whole layer ("per-tensor"): one neuron with large weights then cannot ruin the precision of all the others.

## 3. Quantizing the baseline's weights

Let's look at the weights of the first layer (19 200 numbers):

In [ ]:
W0 = baseline.layers[0].weight.detach()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.3))
a1.hist(W0.flatten().numpy(), bins=100, color="C0")
a1.set_xlabel("weight value"); a1.set_ylabel("count"); a1.set_title("Layer 0: all weights")
a2.bar(range(W0.shape[0]), W0.abs().amax(dim=1).numpy(), color="C0")
a2.set_xlabel("neuron (output channel)"); a2.set_ylabel("max |w|"); a2.set_title("Largest weight per neuron")
fig.tight_layout(); plt.show()

Most weights are small, and the largest weight differs quite a lot from neuron to neuron. Let's write a function that quantizes the weights of every layer (per-tensor or per-channel) and puts the *dequantized* values back, so we can measure what the rounding does to the accuracy. This is called **simulated quantization**: still floats, but only values that an integer could represent.

In [ ]:
def quantize_weights(model, bits=8, per_channel=True):
    """Return a copy of `model` whose weights are rounded to `bits`-bit integers (and back)."""
    qmax = 2 ** (bits - 1) - 1                         # 127 for 8 bits, 7 for 4 bits
    m = copy.deepcopy(model)
    with torch.no_grad():
        for layer in m.layers:
            w = layer.weight
            if per_channel:
                max_abs = w.abs().amax(dim=1, keepdim=True)   # one scale per neuron (row)
            else:
                max_abs = w.abs().max()                       # one scale for the layer
            s = max_abs / qmax
            layer.weight.copy_(torch.clamp(torch.round(w / s), -qmax, qmax) * s)
    return m

print(f"{'bits':>5} {'per-tensor':>12} {'per-channel':>12}   (test accuracy, float32 = {base_acc:.1%})")
for bits in [8, 6, 4, 3, 2]:
    acc_t = accuracy(quantize_weights(baseline, bits, per_channel=False), data.X_test, data.y_test)
    acc_c = accuracy(quantize_weights(baseline, bits, per_channel=True), data.X_test, data.y_test)
    print(f"{bits:>5} {acc_t:>12.1%} {acc_c:>12.1%}")

**Observations**
* With **8 bits the accuracy does not change** (differences of a few tenths of a point are noise). This is why int8 is the industry standard.
* Down to ~4 bits things still work; at 2 bits (only the values -1, 0 and +1 times a scale!) the network breaks.
* At very low bit widths, **per-channel is clearly better** than per-tensor.

## 4. Quantizing the whole network

So far only the *weights* were integers. On the board, the input and the numbers passed between the layers (the **activations**) are int8 as well, so that all multiplications can use integers.

For the weights we know the range: we can just look at them. The activations depend on the input, so we **measure** their range by running the training data through the network once. This is called **calibration**.

Train in float, then calibrate and quantize: this recipe is called **post-training quantization (PTQ)**.

First three small helpers, straight from the formula in section 2:

* `quantize`: real number → integer (divide by the scale, round, add the zero-point, clamp);
* `dequantize`: integer → (approximately) the real number again;
* `fake_quantize`: both in a row. The result is still a float, but it can only be a value that an integer could represent. This is how we *simulate* integer maths while staying in PyTorch.

Ignore the last line of `fake_quantize` for now. It gives the same value as `rounded` and only matters for training, in section 6.

In [ ]:
def quantize(x, scale, zero_point, qmin, qmax):
    """real -> integer"""
    return torch.clamp(torch.round(x / scale) + zero_point, qmin, qmax)

def dequantize(q, scale, zero_point):
    """integer -> real (approximately the original number)"""
    return (q - zero_point) * scale

def fake_quantize(x, scale, zero_point, qmin, qmax):
    """Round to the integer grid and back, so the result is a float that only takes 'integer' values."""
    lo, hi = (qmin - zero_point) * scale, (qmax - zero_point) * scale
    x_clipped = torch.maximum(torch.minimum(x, torch.as_tensor(hi)), torch.as_tensor(lo))
    rounded = dequantize(quantize(x_clipped, scale, zero_point, qmin, qmax), scale, zero_point)
    # Same value as `rounded`, but backpropagation treats the rounding as "do nothing"
    # (the straight-through estimator, explained in section 6).
    return x_clipped + (rounded - x_clipped).detach()

`QuantMLP` wraps our trained network. It does not change the network: every time data goes through, it rounds the numbers the way the board will. Its parts:

* `calibrate(X)` runs the training data through the float network once. It remembers the smallest and largest value (`act_lo`, `act_hi`) at three **measurement points**: the input (point 0), after hidden layer 1 (point 1) and after hidden layer 2 (point 2). Point `k` is always the input of layer `k`. After a ReLU nothing is negative, so the minimum there is simply 0. The output of the last layer (the 5 logits) is not quantized.
* `act_params(k)` turns the range of point `k` into a scale and zero-point, with `asymmetric_params` from section 2.
* `weight_scale(layer)` gives one scale per neuron (per row of the weight matrix): the largest |weight| of that row divided by 127.
* `forward(x)` does what the board does: quantize the input; then, for every layer, round the weights and the bias, compute the layer, apply the ReLU and quantize the result.

Two lines in `forward` deserve a closer look:
* `s_w[:, None]` turns the list of scales into a column, so that every *row* of the weight matrix is divided by its own scale.
* The bias is rounded to a **32-bit** integer, with scale `s_x * s_w`. Section 5 explains why.

Skip the parts marked "section 6" (`ema_momentum` and the `if self.training:` block). They are only used when we train the quantized network.

In [ ]:
class QuantMLP(nn.Module):
    """Wraps a trained MLP and simulates integer inference with fake quantization."""

    def __init__(self, mlp, weight_bits=8, act_bits=8, ema_momentum=0.02):
        super().__init__()
        self.mlp = mlp                                # the trained float network (not changed)
        self.weight_bits, self.act_bits = weight_bits, act_bits
        self.ema_momentum = ema_momentum              # only used while training (section 6)
        n_points = len(mlp.layers)                    # the input + every hidden layer output
        # A buffer is stored and copied with the model, but the optimizer never changes it.
        self.register_buffer("act_lo", torch.zeros(n_points))   # smallest value seen at each point
        self.register_buffer("act_hi", torch.zeros(n_points))   # largest value seen at each point
        self.calibrated = False

    @torch.no_grad()                                  # only measuring: no gradients needed
    def calibrate(self, X):
        """Measure the range of the input and of every hidden activation on real data."""
        self.mlp.eval()
        x = X
        self.act_lo[0], self.act_hi[0] = x.min(), x.max()          # point 0: the input
        for i, layer in enumerate(self.mlp.layers[:-1]):           # [:-1]: the logits are not quantized
            x = torch.relu(layer(x))
            self.act_lo[i + 1], self.act_hi[i + 1] = 0.0, x.max()   # after ReLU the minimum is 0
        self.calibrated = True

    def act_params(self, k):
        """(scale, zero-point) of measurement point k (0 = input, 1 = after hidden layer 1, ...)."""
        return asymmetric_params(self.act_lo[k].item(), self.act_hi[k].item(), self.act_bits)

    def weight_scale(self, layer):
        """One scale per output neuron: its largest |weight| lands on the largest integer."""
        qmax = 2 ** (self.weight_bits - 1) - 1
        # layer.weight has one row per neuron; amax(dim=1) is the largest |weight| of each row.
        # clamp: a neuron with only zero weights would otherwise get scale 0.
        return torch.clamp(layer.weight.abs().amax(dim=1).detach(), min=1e-8) / qmax

    def _act(self, k, x):
        """Fake-quantize an activation. While training, the ranges follow the data slowly."""
        if self.training:                             # QAT only (section 6); accuracy() switches this off
            # Moving average: keep 98% of the old range, take 2% of this batch's range.
            m = self.ema_momentum
            lo = x.min() if k == 0 else torch.zeros(())
            self.act_lo[k] = (1 - m) * self.act_lo[k] + m * lo
            self.act_hi[k] = (1 - m) * self.act_hi[k] + m * x.max()
        scale, zp = self.act_params(k)
        return fake_quantize(x, scale, zp, -(2 ** (self.act_bits - 1)), 2 ** (self.act_bits - 1) - 1)

    def forward(self, x):
        if not self.calibrated:
            raise RuntimeError("call calibrate(X_train) first")
        x = self._act(0, x)                                         # quantize the input
        qmax_w = 2 ** (self.weight_bits - 1) - 1
        n = len(self.mlp.layers)
        for i, layer in enumerate(self.mlp.layers):
            s_w = self.weight_scale(layer)                          # shape (outputs,): one scale per neuron
            # s_w[:, None] has shape (outputs, 1), so every row is divided by its own scale.
            # Symmetric: zero-point 0 and range -127...127.
            w = fake_quantize(layer.weight, s_w[:, None], 0, -qmax_w, qmax_w)
            s_x, _ = self.act_params(i)                             # scale of this layer's input
            b = fake_quantize(layer.bias, s_x * s_w, 0, -(2 ** 31), 2 ** 31 - 1)   # int32 bias
            x = F.linear(x, w, b)                                   # x @ w.T + b
            if i < n - 1:
                x = self._act(i + 1, torch.relu(x))                 # ReLU, then quantize again
        return x

In [ ]:
ptq = QuantMLP(copy.deepcopy(baseline), weight_bits=8, act_bits=8)
ptq.calibrate(data.X_train)        # measure activation ranges on the training data

for k in range(len(ptq.act_lo)):
    scale, zp = ptq.act_params(k)
    name = "input" if k == 0 else f"after hidden layer {k}"
    print(f"{name:22s} range [{ptq.act_lo[k]:6.2f}, {ptq.act_hi[k]:6.2f}]  ->  scale {scale:.4f}, zero-point {zp}")

ptq_acc = accuracy(ptq, data.X_test, data.y_test)
print(f"\nPTQ int8 (simulated) test accuracy: {ptq_acc:.1%}   (float32: {base_acc:.1%})")

## 5. How does a layer compute with integers?

`QuantMLP` only *simulates* integers: it still computes with floats. On the board, a layer really works with integers. How is that possible?

**Step 1: the products.** Every quantized number is a scale times an integer: a weight $w \approx s_w\, q_w$ and an input $x \approx s_x\,(q_x - z_x)$. So one product is

$$w \cdot x \approx s_w s_x \cdot q_w\,(q_x - z_x)$$

**Step 2: the sum.** A neuron adds up 300 (or 64, or 32) of these products. The scales $s_w s_x$ are the same in all of them, so we can add up the *integer* parts first and multiply by the scales only once, at the end:

$$y = \sum w\,x + b \;\approx\; s_w s_x \cdot \text{acc} \qquad\text{with}\qquad \text{acc} = q_b + \sum q_w\,(q_x - z_x)$$

The sum `acc` (the **accumulator**) uses only integer multiplications and additions. These are the multiply-accumulates (MACs): the expensive part. It must be a 32-bit integer, because 300 products of up to 127 × 255 add up to millions.

The bias is added directly to `acc`, so it must be in the same units: $q_b = \text{round}\big(b / (s_w s_x)\big)$. The scale $s_w s_x$ is tiny, so $q_b$ is often a large number. That is why the bias is stored as a 32-bit integer.

**Step 3: back to int8.** The next layer wants its input as an int8, on its *own* ruler with scale $s_y$ and zero-point $z_y$. One unit of `acc` is worth $s_w s_x$, one step on the new ruler is worth $s_y$, so:

$$q_y = \text{round}(M \cdot \text{acc}) + z_y \qquad\text{with}\qquad M = \frac{s_w\, s_x}{s_y}$$

$M$ packs the three scales into one number. It costs one float multiplication per *neuron*, not per weight, so almost nothing. Every neuron has its own weight scale, so every neuron has its own $M$ (in formulas often written $M_j$ for neuron $j$).

**Step 4: the ReLU.** Finally $q_y$ is clamped to the range $z_y \dots 127$. Remember that $z_y$ is the integer that stands for real 0. Everything below it was negative, so this clamp **is the ReLU**.

`export_int8` collects everything the board needs from the calibrated model: for every layer the **int8 weights**, the **int32 biases**, the zero-points and the multipliers $M$.

In [ ]:
@torch.no_grad()
def export_int8(qmodel):
    """Collect the integer model: int8 weights, int32 biases, scales, zero-points and multipliers."""
    layers = []
    n = len(qmodel.mlp.layers)
    qmax_w = 2 ** (qmodel.weight_bits - 1) - 1
    for i, layer in enumerate(qmodel.mlp.layers):
        s_x, z_x = qmodel.act_params(i)                       # scale / zero-point of this layer's input
        s_w = qmodel.weight_scale(layer).double()             # one weight scale per neuron
        w_q = quantize(layer.weight.double(), s_w[:, None], 0, -qmax_w, qmax_w)
        b_q = torch.round(layer.bias.double() / (s_x * s_w))  # bias scale = s_x * s_w
        entry = {"w_q": w_q.numpy().astype(np.int8),
                 "b_q": b_q.numpy().astype(np.int32),
                 "in_zero_point": int(z_x)}
        if i < n - 1:
            s_y, z_y = qmodel.act_params(i + 1)               # scale / zero-point of the output
            entry["out_zero_point"] = int(z_y)
            entry["multiplier"] = (s_x * s_w.numpy() / s_y).astype(np.float32)   # M_j
        else:
            entry["multiplier"] = (s_x * s_w.numpy()).astype(np.float32)         # last layer: real logits
        layers.append(entry)
    s_in, z_in = qmodel.act_params(0)
    return {"layers": layers, "input_scale": np.float32(s_in), "input_zero_point": int(z_in)}

`int8_forward` runs the network with NumPy *integers*, following steps 1 to 4 above. It is the Python twin of the C++ code on the Arduino: notebook 05 checks that both give *identical* results. Two details:
* `round_half_away` rounds like C's `roundf` (2.5 → 3, −2.5 → −3). Python's own `round` would round 2.5 to 2, and the results would no longer match the board exactly.
* The `assert` checks that the accumulator is small enough to be converted to float32 without rounding.

In [ ]:
def round_half_away(v):
    """Round like C's roundf: halves go away from zero (2.5 -> 3, -2.5 -> -3)."""
    v = v.astype(np.float64)
    return np.sign(v) * np.floor(np.abs(v) + 0.5)


def int8_forward(int8_model, x):
    """int8 inference: integer MACs, one float rescale per neuron. x: normalized float features (N, 300). Returns float logits (N, 5)."""
    # quantize the input to int8
    a = round_half_away(np.asarray(x, dtype=np.float32) / int8_model["input_scale"]) + int8_model["input_zero_point"]
    a = np.clip(a, -128, 127).astype(np.int64)
    layers = int8_model["layers"]
    for i, L in enumerate(layers):
        # acc = bias + sum of w_q * (x_q - z_x): pure integer multiply-accumulate
        acc = L["b_q"].astype(np.int64) + (a - L["in_zero_point"]) @ L["w_q"].astype(np.int64).T
        assert np.abs(acc).max() < 2 ** 24, "accumulator too large for exact float32 conversion"
        p = acc.astype(np.float32) * L["multiplier"]          # one float multiply per neuron
        if i < len(layers) - 1:
            q = round_half_away(p) + L["out_zero_point"]
            a = np.clip(q, L["out_zero_point"], 127).astype(np.int64)   # the lower clamp is the ReLU
        else:
            return p.astype(np.float32)


def int8_accuracy(int8_model, X, y):
    return float((int8_forward(int8_model, X.numpy()).argmax(1) == y.numpy()).mean())


def int8_model_bytes(int8_model):
    """Flash memory: int8 weights + int32 biases + one float32 multiplier per neuron."""
    return sum(L["w_q"].size + 4 * L["b_q"].size + 4 * L["multiplier"].size for L in int8_model["layers"])

In [ ]:
int8_model = export_int8(ptq)
L0 = int8_model["layers"][0]
print("layer 0 weights :", L0["w_q"].dtype, L0["w_q"].shape, " e.g.", L0["w_q"][0, :8])
print("layer 0 biases  :", L0["b_q"].dtype, L0["b_q"].shape, " e.g.", L0["b_q"][:4])
print("layer 0 M       :", L0["multiplier"].dtype, " e.g.", L0["multiplier"][:3])

logits_int = int8_forward(int8_model, data.X_test.numpy())
with torch.no_grad():
    logits_sim = ptq(data.X_test).numpy()
print(f"\nint8 inference accuracy: {int8_accuracy(int8_model, data.X_test, data.y_test):.1%}")
print(f"same predictions as the simulation: {(logits_int.argmax(1) == logits_sim.argmax(1)).mean():.1%}"
      f"   (largest logit difference {np.abs(logits_int - logits_sim).max():.1e})")
print(f"memory: {int8_model_bytes(int8_model) / 1024:.1f} KB  (float32: {4 * count_params(baseline) / 1024:.1f} KB)")

So the int8 network is (practically) **as accurate as the float network, at about 1/4 of the memory**. The few extra bytes come from the 32-bit biases and one multiplier per neuron.

## 6. Fewer bits: quantization-aware training (QAT)

At 8 bits, PTQ is enough. But in section 3 we saw that at 2 bits the network breaks. The fix: let the network **experience the rounding during training**, so it can learn weights that still work after rounding. This is **quantization-aware training (QAT)**.

We simply train the `QuantMLP` with the normal training loop. Every forward pass rounds the weights and activations, so the network sees the same errors as on the board. Two tricks make that work:
* **Rounding has no useful slope.** `round()` is a staircase: flat everywhere, so its gradient is 0 and nothing would be learned. The **straight-through estimator (STE)** pretends, during backpropagation only, that the rounding is not there. That is the last line of `fake_quantize`.
* **The activation ranges move**, because the weights change. That is the `if self.training:` block in `_act`: after every batch, the stored range moves 2% towards the range of that batch (a *moving average*), so it slowly follows the data.

We start from the trained baseline and train a few more epochs. To be fair, we also train the *float* model for the same number of extra epochs (the "control"). That shows how much of any improvement just comes from the extra training. The activations stay 8-bit throughout.

In [ ]:
bit_widths = [8, 4, 3, 2]
results = {"PTQ": [], "QAT": []}
set_seed(0)
control = copy.deepcopy(baseline)
train(control, data, lr=5e-4, epochs=30, patience=10, augment=True, verbose=False)
control_acc = accuracy(control, data.X_test, data.y_test)

for bits in bit_widths:
    q = QuantMLP(copy.deepcopy(baseline), weight_bits=bits)
    q.calibrate(data.X_train)
    results["PTQ"].append(accuracy(q, data.X_test, data.y_test))
    train(q, data, lr=5e-4, epochs=30, patience=10, augment=True, verbose=False)   # <- this is QAT
    results["QAT"].append(accuracy(q, data.X_test, data.y_test))
    print(f"{bits} bits: PTQ {results['PTQ'][-1]:.1%}   QAT {results['QAT'][-1]:.1%}")
print(f"float32 baseline {base_acc:.1%}, float32 fine-tuned (control) {control_acc:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8))
for name, marker in zip(results, ["o", "s"]):
    ax.plot(bit_widths, results[name], marker=marker, markersize=6, label=name)
ax.axhline(base_acc, color="gray", ls=":", lw=1.5)
ax.text(bit_widths[-1], base_acc, "  float32 baseline", va="bottom", color="gray")
ax.set_xlabel("weight bits"); ax.set_ylabel("test accuracy"); ax.set_title("Accuracy vs weight bit width")
ax.legend(); ax.invert_xaxis(); plt.show()

**What do you see?**
* At **8 and 4 bits**, PTQ, QAT and the float control are all within about a point of each other: noise (remember notebook 01). For int8, PTQ is all you need.
* At **2 bits**, PTQ collapses, while QAT brings the accuracy back close to the float model. The network has *learned* to live with only three weight values per neuron (−1, 0 and +1 times a scale).

Without the control, we might have concluded that QAT also improves 8-bit models!

Fewer bits only save memory if several weights are **packed** into one byte (two 4-bit or four 2-bit weights), and the C code must then unpack them again. The Arduino code in this course uses int8: the standard choice, supported directly by the hardware.

## 7. Other kinds of quantization (overview)

What we built is **full-integer post-training quantization**: int8 weights *and* int8 activations, with the activation ranges found by calibration. Toolkits such as TensorFlow Lite (LiteRT, via its `TFLiteConverter`) and PyTorch offer a few other variants. You will meet their names in tutorials:

| Variant | Weights | Activations | Calibration data? | Typical use |
|---|---|---|---|---|
| **Full integer** (this notebook) | int8 | int8 | yes | microcontrollers, Edge TPU, NPUs |
| **Dynamic-range** | int8 | float; quantized on the fly in every layer | no | a quick first try on phones and PCs |
| **float16** | float16 | float | no | GPUs; 2x smaller, but no speed gain on our board |
| **16x8** | int8 | int16 | yes | audio and image clean-up, where the exact output values matter |

The column *Calibration data?* asks whether you need a set of representative examples (typically 100-500) to measure the activation ranges, as in section 4. Only variants that store activations as integers need it.

**Full integer** is what this notebook built. Weights and activations are int8, and all MACs run on integers. Because the activation ranges are measured once, in advance, every scale and every $M$ is a fixed number that can be stored with the model. That is what a microcontroller needs, and it also suits accelerators such as the Edge TPU and NPUs (neural processing units), which only compute with integers.

**Dynamic-range** quantization stores only the *weights* as int8. The activations stay float, so no calibration is needed. Instead, each time a layer runs, it looks at its input for *this* prediction, picks a scale on the spot, quantizes the input, does the MACs with integers and converts the result back to float. The bias and the ReLU stay float.
* **+** Very easy to try: no example data needed, the model is 4x smaller, and the MACs still run on integers.
* **−** Measuring the range costs extra work in every layer, every time. The chip still needs float arithmetic, so it does not work on integer-only hardware.

**float16** ("half precision") is a 16-bit floating-point number instead of a 32-bit one. Converting a model just means storing every weight with fewer digits: no scale, no zero-point, no calibration.
* A float16 keeps only about 3 significant decimal digits, and its largest value is 65 504. That is plenty for weights, so the accuracy practically does not change, and the model is 2x smaller.
* GPUs and some phone chips compute in float16 directly, and fast. Our Cortex-M4F cannot: it would convert every weight back to float32 before using it. A smaller model, but not a faster one.

**16x8** keeps int8 weights but uses **int16 activations**: 65 536 levels instead of 256. This helps networks whose *output is itself a signal*, such as noise cancelling, cleaning up images or making images sharper. There, a small rounding error in every output value is audible or visible. The price:
* the activations take twice as much memory;
* the sums get much larger, so toolkits use 64-bit integers for the biases and the accumulator;
* fewer chips have fast int16 × int8 instructions, so it is usually slower than int8.

For a classifier like ours, 16x8 brings nothing: only *which* output is largest matters, and tiny rounding errors do not change that.

A few more terms:
* **QAT** (section 6) can be combined with these variants. Use it when post-training quantization loses too much accuracy.
* **Partial** or **mixed-precision** quantization: keep the most sensitive layers at higher precision (or in float) and quantize the rest. Measure first which layer suffers most, then spend the bits where they matter.
* **Integer-only** hardware (no FPU, like the Edge TPU) also needs integer inputs and outputs, and replaces our float multiplication by $M$ with an integer multiplication plus a bit shift. Our Cortex-M4F has an FPU, so we keep the float multiplication: it is easier to read.

## Summary

* Quantization maps real numbers to integers: $q = \text{round}(r/s) + z$, and back: $r \approx s\,(q - z)$.
* Weights: symmetric, one scale per neuron (per-channel). Activations: asymmetric, with ranges found by **calibration**.
* A layer then runs on **integer MACs**, plus one float multiplication per neuron.
* **int8 PTQ** makes the model 4x smaller at essentially no accuracy cost.
* At very low bit widths, **QAT** (fake quantization + straight-through estimator) recovers the accuracy.
* Toolkits offer other variants (dynamic-range, float16, 16x8, mixed precision); microcontrollers use full-integer int8.

## Exercises

1. ⭐ **By hand.** Quantize the weights `[0.9, -0.3, 0.02, -1.8]` to 4 bits (symmetric, $q \in [-7, 7]$) on paper. What are $s$, the $q$'s, the dequantized values and the largest error? Check with code.
2. ⭐ **Outliers.** Take a copy of the baseline and set *one* weight of layer 0 to 20.0. Compare per-tensor vs per-channel 4-bit accuracy (`quantize_weights`). Explain the result with the "ruler" picture.
3. **Activation bits.** Try `QuantMLP(..., act_bits=4)` with weights at 8 bits. (`export_int8` and the board only support 8-bit activations, but the simulation works.) Which is more sensitive, weights or activations?
4. **Calibration data.** Calibrate the PTQ model on only the first 10 training examples instead of all. What happens to the ranges and the accuracy? Why?
5. **Smaller models.** Repeat the bit-width experiment for a small `MLP(hidden=(16, 8))`. Do small networks tolerate quantization better or worse? Why could that be?